## Feature Importance Models
This file runs random forest and gradient boosting classification on the final version of the data (master_df.csv)

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

MASTER_DF_PATH = './master_df_hs.csv'

In [11]:
data = pd.read_csv(MASTER_DF_PATH)
# List the current columns
print(data.columns)

Index(['LEA_STATE', 'LEA_STATE_NAME', 'LEAID', 'LEA_NAME', 'SCHID', 'SCH_NAME',
       'COMBOKEY', 'total_enrollment_male', 'total_enrollment_female',
       'teacher_salary_expend', 'teacher_salary_expend_fed', 'teachers_fte',
       'num_curr_year_teachers', 'num_prev_year_teachers',
       'num_teacher_absences', 'num_counselors', 'student_teacher_ratio',
       'average_teacher_salary', 'average_teacher_salary_adjusted',
       'suspensions_male', 'suspensions_female', 'num_advanced_math_courses',
       'num_ap_courses', 'ap_enrollment', 'num_bio_courses',
       'num_calc_courses', 'num_chem_courses', 'num_geo_courses',
       'num_phys_courses', 'num_sat_act_students', 'harassment_sex',
       'harassment_race', 'harassment_disability', 'harassment_orientation',
       'harassment_religion', 'chronic_absences_male',
       'chronic_absences_female', 'combined_proficiency_score_2018',
       'reading_proficiency_score_2018', 'reading_proficiency_class',
       'math_proficiency_s

In [12]:
# Load the data
data = pd.read_csv(MASTER_DF_PATH)

# Define the features and target
numeric_features = numeric_features = [
    'total_enrollment_male',
    'total_enrollment_female',
    'teacher_salary_expend', 
    'teacher_salary_expend_fed',
    'teachers_fte', 
    'num_teacher_absences',
    'num_counselors',
    'num_curr_year_teachers',
    'num_prev_year_teachers',
    'student_teacher_ratio',
    'average_teacher_salary_adjusted', 
    'suspensions_male',
    'suspensions_female',
    'harassment_sex',
    'harassment_race',
    'harassment_disability',
    'harassment_orientation',
    'harassment_religion',
    'chronic_absences_male',
    'chronic_absences_female',
    'num_advanced_math_courses',
    'num_ap_courses',
    'ap_enrollment',
    'num_bio_courses',
    'num_calc_courses',
    'num_chem_courses',
    'num_geo_courses',
    'num_phys_courses',
    'num_sat_act_students'
]

X = data[numeric_features]


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from pprint import pprint

def generate_random_forest_hyperparam_grid():
    # Number of trees in the forest
    n_estimators = [int(x) for x in np.linspace(start=100, stop=300, num=9)]
    
    # Number of features to consider at every split
    max_features = ['log2', 'sqrt', None]
    
    # Maximum number of levels in each tree
    max_depth = [int(x) for x in np.linspace(10, 110, num=11)]
    max_depth.append(None)
    
    # Minimum number of samples required to split a node
    min_samples_split = [2, 5, 10]
    
    # Minimum number of samples required at each leaf node
    min_samples_leaf = [1, 2, 4]
    
    # Method of selecting samples for training each tree
    bootstrap = [True, False]
    
    # Create the random grid
    random_grid = {
        'n_estimators': n_estimators,
        'max_features': max_features,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'bootstrap': bootstrap
    }
    
    pprint(random_grid)
    return random_grid

# Generate and print the grid
random_forest_grid = generate_random_forest_hyperparam_grid()

{'bootstrap': [True, False],
 'max_depth': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, None],
 'max_features': ['log2', 'sqrt', None],
 'min_samples_leaf': [1, 2, 4],
 'min_samples_split': [2, 5, 10],
 'n_estimators': [100, 125, 150, 175, 200, 225, 250, 275, 300]}


In [14]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from pprint import pprint

def generate_gradient_boosting_hyperparam_grid():
    # Number of trees in the boosting process
    n_estimators = [int(x) for x in np.linspace(start=100, stop=300, num=9)]
    
    # Maximum number of levels in each tree
    max_depth = [int(x) for x in np.linspace(3, 10, num=8)]
    max_depth.append(None)
    
    # Learning rate (step size shrinkage)
    learning_rate = [0.001, 0.01, 0.1, 0.05, 0.2]
    
    # Minimum number of samples required to split a node
    min_samples_split = [2, 5, 10]
    
    # Minimum number of samples required at each leaf node
    min_samples_leaf = [1, 2, 4]
    
    # Maximum number of features to consider for the best split
    max_features = ['log2', 'sqrt', None]
    
    # Subsample ratio of the training instance
    subsample = [0.6, 0.8, 1.0]
    
    # Create the random grid
    random_grid = {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'max_features': max_features,
        'subsample': subsample
    }
    
    pprint(random_grid)
    return random_grid

# Generate and print the grid
gradient_boosting_grid = generate_gradient_boosting_hyperparam_grid()

{'learning_rate': [0.001, 0.01, 0.1, 0.05, 0.2],
 'max_depth': [3, 4, 5, 6, 7, 8, 9, 10, None],
 'max_features': ['log2', 'sqrt', None],
 'min_samples_leaf': [1, 2, 4],
 'min_samples_split': [2, 5, 10],
 'n_estimators': [100, 125, 150, 175, 200, 225, 250, 275, 300],
 'subsample': [0.6, 0.8, 1.0]}


In [18]:
def hyperparam_tune(model, param_grid, n_iter=5, random_state=42):
    model_random = RandomizedSearchCV(estimator = model, param_distributions = param_grid, n_iter = n_iter, cv = 3, verbose=2, random_state=random_state, n_jobs = -1)
    return model_random

In [15]:
def train_model(X, y, classifier, hyperparam_grid, n_iters=5, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=111)
    model = hyperparam_tune(classifier(), hyperparam_grid, n_iters, random_state=random_state)
    # Fit the model on the training data
    model.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test)

    # Calculate and print evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    print("Accuracy:", accuracy)

    # Additional evaluation metrics
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    return model.best_estimator_

In [16]:
def feature_importance(model):
    # Get feature importances
    feature_importances = model.feature_importances_

    # Create a DataFrame for easier visualization
    importance_df = pd.DataFrame({
        'Feature': numeric_features,
        'Importance': feature_importances
    }).sort_values(by='Importance', ascending=False)

    print(importance_df)

### Random Forest Classifier - Reading Scores

In [19]:
y = data['reading_proficiency_class']
model = train_model(X, y, RandomForestClassifier, random_forest_grid, 50)
feature_importance(model)
pprint(model.get_params())

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END bootstrap=True, max_depth=40, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=225; total time=   5.1s
[CV] END bootstrap=True, max_depth=40, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=225; total time=   5.1s
[CV] END bootstrap=True, max_depth=40, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=225; total time=   5.3s
[CV] END bootstrap=False, max_depth=110, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=300; total time=   8.8s
[CV] END bootstrap=False, max_depth=100, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=250; total time=   8.9s
[CV] END bootstrap=False, max_depth=100, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=250; total time=   9.2s
[CV] END bootstrap=False, max_depth=110, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators

### Random Forest Classifier - Math Scores

In [20]:
y = data['math_proficiency_class']
model = train_model(X, y, RandomForestClassifier, random_forest_grid, 50)
feature_importance(model)
pprint(model.get_params())

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END bootstrap=True, max_depth=40, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=225; total time=   4.9s
[CV] END bootstrap=True, max_depth=40, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=225; total time=   4.9s
[CV] END bootstrap=True, max_depth=40, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=225; total time=   5.0s
[CV] END bootstrap=False, max_depth=110, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=300; total time=  11.5s
[CV] END bootstrap=False, max_depth=110, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=300; total time=  11.8s
[CV] END bootstrap=False, max_depth=110, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=300; total time=  11.9s
[CV] END bootstrap=False, max_depth=100, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators

### Gradient Boost Classifier - Reading Scores

In [21]:
y = data['reading_proficiency_class']
model = train_model(X, y, GradientBoostingClassifier, gradient_boosting_grid, 50)
feature_importance(model)
pprint(model.get_params())

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END learning_rate=0.001, max_depth=4, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=275, subsample=1.0; total time=   8.1s
[CV] END learning_rate=0.001, max_depth=4, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=275, subsample=1.0; total time=   8.1s
[CV] END learning_rate=0.001, max_depth=4, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=275, subsample=1.0; total time=   8.3s
[CV] END learning_rate=0.1, max_depth=6, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100, subsample=0.6; total time=  15.9s
[CV] END learning_rate=0.1, max_depth=6, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100, subsample=0.6; total time=  16.0s
[CV] END learning_rate=0.1, max_depth=6, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100, subsample=0.6; total time=  16.1s
[CV] END learn

### Gradient Boost Classifier - Math Scores

In [22]:
y = data['math_proficiency_class']
model = train_model(X, y, GradientBoostingClassifier, gradient_boosting_grid, 50)
feature_importance(model)
pprint(model.get_params())

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END learning_rate=0.001, max_depth=4, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=275, subsample=1.0; total time=   8.5s
[CV] END learning_rate=0.001, max_depth=4, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=275, subsample=1.0; total time=   8.5s
[CV] END learning_rate=0.001, max_depth=4, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=275, subsample=1.0; total time=   8.6s
[CV] END learning_rate=0.1, max_depth=6, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100, subsample=0.6; total time=  15.4s
[CV] END learning_rate=0.1, max_depth=6, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100, subsample=0.6; total time=  15.5s
[CV] END learning_rate=0.1, max_depth=6, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100, subsample=0.6; total time=  15.7s
[CV] END learn